In [ ]:
!pip install langchain-openai

Defaulting to user installation because normal site-packages is not writeable


In [ ]:
llm_fast_high = "gpt-4o"
llm_fast = "gpt-4o-mini"
llm_smart = "gpt-4o"

In [ ]:
from langchain_openai import ChatOpenAI
print("LangChain OpenAI installed")

LangChain OpenAI installed


In [ ]:
!pip install openai
!pip install --upgrade typing_extensions
!pip install --upgrade typing_extensions
!pip install --upgrade --force-reinstall typing_extensions
from langchain_openai import ChatOpenAI
from langchain_openai import ChatOpenAI
print("LangChain OpenAI installed")

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)
  Attempting uninstall: typing_extensions
    Found existing installation: typing_extensions 4.15.0
    Uninstalling typing_extensions-4.15.0:
      Successfully uninstalled typing_extensions-4.15.0
LangChain OpenAI installed


In [ ]:
!pip install langgraph
!pip install langchain-google-genai

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [ ]:
from langchain_openai import ChatOpenAI
!pip install langchain-anthropic
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langgraph.checkpoint.memory import MemorySaver

Defaulting to user installation because normal site-packages is not writeable


# Agentic Curriculum Builder (Base Prototype)


In [ ]:
# If you need to install some packages
# !pip install langgraph langchain langchain-openai
# !pip install langchain_anthropic langchain_google_genai

import operator
from typing import Annotated, TypedDict, List
from langgraph.graph import StateGraph, END
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage

from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_google_genai import ChatGoogleGenerativeAI

from langgraph.checkpoint.memory import MemorySaver
import os
os.environ["OPENAI_API_KEY"] = ""

In [ ]:
# 0. Initialize Memory for the agent
memory = MemorySaver()

# 1a. Define the Shared State for the agent
# These are the "notes" that get passed between nodes.
class AgentState(TypedDict):
    initial_topic: str
    brainstorm: str
    draft: str
    # This 'operator.add' is the secret sauce for parallel work
    reviewer_feedback: Annotated[List[str], operator.add]
    editor_feedback: str
    revised_draft: str
    final_content: str

    revision_count: int

# 1b. Define the AI models available for the agent
llm_nano      = ChatOpenAI(model="gpt-5-nano", temperature=0.2)
llm_nano_high = ChatOpenAI(model="gpt-5-nano", temperature=0.8)
llm_5         = ChatOpenAI(model="gpt-5",      temperature=0.2)
llm_5_high    = ChatOpenAI(model="gpt-5",      temperature=0.8)


In [ ]:
# 0. Initialize Memory for the agent
memory = MemorySaver()

# 1a. Define the Shared State for the agent
class AgentState(TypedDict):
    initial_topic: str
    brainstorm: str
    draft: str
    reviewer_feedback: Annotated[List[str], operator.add]
    editor_feedback: str
    revised_draft: str
    final_content: str
    revision_count: int
    exam_content: str

# 1b. Define the AI models available for the agent
llm_nano      = ChatOpenAI(model="gpt-5-nano", temperature=0.2)
llm_nano_high = ChatOpenAI(model="gpt-5-nano", temperature=0.8)
llm_5         = ChatOpenAI(model="gpt-5",      temperature=0.2)
llm_5_high    = ChatOpenAI(model="gpt-5",      temperature=0.8)

# ✅ ADD THESE THREE LINES:
llm_fast_high = ChatOpenAI(model="gpt-4o",      temperature=0.8)
llm_fast      = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
llm_smart     = ChatOpenAI(model="gpt-4o",      temperature=0.2)

In [ ]:
initial_topic = "Masked Auto Encoder (MAE) for GeoAI"

# ── Rubric reference (5 criteria × 5 pts = 25 pts per output type) ──────────
# NOTEBOOK:   1-Executability  2-Data Accessibility  3-Content Accuracy
#             4-Instructional Design  5-Clarity & Presentation
# ASSESSMENT: 1-Question Quality  2-Answer Key & Grading  3-Content Accuracy
#             4-Instructional Design  5-Clarity & Presentation

NODE_CONFIGS = {

    # ── BRAINSTORMER ─────────────────────────────────────────────────────────
    # Feeds writer. Quality here determines rubric §3 Content Accuracy ceiling.
    "brainstormer": {
        "system_role": "GeoAI Research Specialist",
        "system_instruction": (
            "Output EXACTLY 7 novel GeoAI lab ideas. No generic concepts.\n\n"
            "Each idea — one block, this format only:\n"
            "Title: <name>\n"
            "Concept: <2 sentences, technically specific>\n"
            "Data: <named dataset: Sentinel-2, LiDAR, MODIS, SAR, etc.>\n"
            "CRS: <coordinate reference system and justification>\n"
            "Method: <model or pipeline>\n"
            "Application: <1 sentence real-world use>\n"
            "Novelty: <1 sentence — what makes this non-obvious>\n\n"
            "CRS must be stated for every idea — this is a hard requirement."
        ),
        "human_instruction": "Topic:",
        "llm": llm_fast_high,
        "output_key": "brainstorm"
    },

    # ── WRITER ───────────────────────────────────────────────────────────────
    # Primary producer. Targets rubric §1 Executability + §2 Data + §4 Design.
    "writer": {
        "system_role": "GeoAI Educator",
        "system_instruction": (
            "Write a Jupyter Notebook lab completable in ~9 minutes.\n"
            "Alternate Markdown Cell / Code Cell throughout.\n\n"
            "REQUIRED SECTIONS (in order):\n"
            "1. Learning Objectives — exactly 3 measurable bullets\n"
            "2. Concept Overview — define all technical terms on first use; "
               "explain concepts BEFORE the code that uses them\n"
            "3. Data — programmatic download cell (no manual steps); cite: "
               "name, provider, URL, access date, license; describe fields, "
               "CRS, units, time period, study area; state dataset size\n"
            "4. Implementation — step-by-step code; every cell commented; "
               "state and justify CRS; set random seed; state expected runtime\n"
            "5. Results — interpret outputs geospatially; maps must have "
               "legend, CRS label, and justified symbology\n"
            "6. Conclusion — restate objectives met; 1 extension suggestion\n\n"
            "CODE RULES:\n"
            "- Executes top-to-bottom on a fresh runtime, zero errors\n"
            "- All imports present; use relative or URL-based paths only\n"
            "- Random seed set for any stochastic operation\n"
            "- Difficulty: beginner–intermediate; no copy-paste exercises"
        ),
        "human_instruction": "Pick the BEST idea and write the full notebook.\nBrainstorm:",
        "llm": llm_fast,
        "output_key": "draft"
    },

    # ── REVIEWER A — Geospatial Accuracy (Rubric §3 Content Accuracy) ────────
    "reviewer_a": {
        "system_role": "GIScientist Reviewer",
        "system_instruction": (
            "Audit against rubric §3 Content Accuracy. Flag every violation:\n"
            "- Geospatial concepts incorrectly defined or misapplied\n"
            "- CRS not stated, unjustified, or inconsistent across datasets\n"
            "- Spatial/analytical methods ill-suited to the data or question\n"
            "- Maps missing legend, CRS label, or unjustified symbology\n"
            "- Model choice unjustified; spatial autocorrelation in splits "
              "not addressed; results not interpreted geospatially\n\n"
            "For each issue:\n"
            "Issue: <what is wrong>\n"
            "Rubric: §3.<sub-criterion>\n"
            "Fix: <exact correction>"
        ),
        "human_instruction": "Notebook draft:",
        "llm": llm_fast,
        "output_key": "reviewer_a_feedback"
    },

    # ── REVIEWER B — Code & Data Quality (Rubric §1 + §2) ───────────────────
    "reviewer_b": {
        "system_role": "ML Engineer Reviewer",
        "system_instruction": (
            "Audit against rubric §1 Executability and §2 Data Accessibility.\n\n"
            "§1 checks:\n"
            "- Will it run top-to-bottom on a fresh runtime without errors?\n"
            "- Are all imports present?\n"
            "- Are all paths relative/URL-based (no absolute local paths)?\n"
            "- Do expected outputs display correctly (not blank/error)?\n"
            "- Is a random seed set? Is expected runtime stated?\n\n"
            "§2 checks:\n"
            "- Is data retrieved programmatically (no manual download)?\n"
            "- Is source fully cited: name, provider, URL/DOI, date, license?\n"
            "- Is dataset structure described: fields, CRS, units, time, area?\n"
            "- Is dataset size appropriate for teaching?\n\n"
            "For each issue:\n"
            "Issue: <what is wrong>\n"
            "Rubric: §<1 or 2>.<sub-criterion>\n"
            "Fix: <exact correction>"
        ),
        "human_instruction": "Notebook draft:",
        "llm": llm_fast,
        "output_key": "reviewer_b_feedback"
    },

    # ── REVIEWER C — Pedagogy & Clarity (Rubric §4 + §5) ────────────────────
    "reviewer_c": {
        "system_role": "Instructional Design Reviewer",
        "system_instruction": (
            "Audit against rubric §4 Instructional Design and §5 Clarity.\n\n"
            "§4 checks:\n"
            "- Are concepts explained before the code that uses them?\n"
            "- Is there a logical, step-by-step progression?\n"
            "- Are technical terms defined on first use?\n"
            "- Is difficulty appropriate for the target audience?\n"
            "- Does the lab engage higher-order thinking (not just copy-paste)?\n\n"
            "§5 checks:\n"
            "- Are instructions clear and concise?\n"
            "- Is every code cell commented with an explanation?\n"
            "- Are learning objectives and conclusion clearly stated?\n"
            "- Is formatting consistent throughout?\n"
            "- Is the overall flow easy to follow start to finish?\n\n"
            "For each issue:\n"
            "Issue: <what is wrong>\n"
            "Rubric: §<4 or 5>.<sub-criterion>\n"
            "Fix: <exact improvement>"
        ),
        "human_instruction": "Notebook draft:",
        "llm": llm_fast,
        "output_key": "reviewer_c_feedback"
    },

    # ── EDITOR ───────────────────────────────────────────────────────────────
    # Synthesises all three rubric-mapped reviews into a ranked action plan.
    "editor": {
        "system_role": "Curriculum Editor",
        "system_instruction": (
            "Synthesise the three rubric reviews into a decisive revision plan.\n\n"
            "TOP 5 PRIORITIES (highest rubric impact first):\n"
            "1.\n2.\n3.\n4.\n5.\n\n"
            "REVISION PLAN (table):\n"
            "Rubric Section | Issue | Required Action\n\n"
            "Resolve any conflicts between reviewers. Every action must map "
            "to a specific rubric criterion. Be concrete — no vague edits."
        ),
        "human_instruction": "Draft + reviews:",
        "llm": llm_smart,
        "output_key": "editor_feedback"
    },

    # ── REVISER ──────────────────────────────────────────────────────────────
    # Full rewrite implementing the editor plan; must pass all 5 rubric §s.
    "reviser": {
        "system_role": "GeoAI Author",
        "system_instruction": (
            "Rewrite the notebook to implement every item in the editor plan.\n\n"
            "Before submitting, self-check each rubric criterion:\n"
            "§1 Executability — fresh-runtime runnable, seeds set, runtime stated\n"
            "§2 Data — programmatic download, full citation, structure described\n"
            "§3 Content Accuracy — CRS stated/justified/consistent, methods valid, "
               "maps labeled, results interpreted geospatially\n"
            "§4 Instructional Design — concepts before code, terms defined, "
               "logical progression, higher-order tasks\n"
            "§5 Clarity — objectives + conclusion present, all cells commented, "
               "consistent formatting, easy flow\n\n"
            "Substantial rewrite where needed. Output must be cohesive."
        ),
        "human_instruction": "Editor plan + original draft:",
        "llm": llm_smart,
        "output_key": "revised_draft"
    },

    # ── COPYWRITER ───────────────────────────────────────────────────────────
    # Final polish targeting §5 Clarity & Presentation specifically.
    "copywriter": {
        "system_role": "Technical Copyeditor",
        "system_instruction": (
            "Final pass targeting rubric §5 Clarity & Presentation.\n\n"
            "Check and fix:\n"
            "- Instructions clear and concise (no redundant sentences)\n"
            "- Every code cell has an explanatory comment\n"
            "- Learning objectives stated at top; conclusion at bottom\n"
            "- Formatting consistent: heading levels, spacing, code style\n"
            "- Flow reads naturally start to finish\n\n"
            "Do NOT alter technical content, CRS choices, or data citations."
        ),
        "human_instruction": "Revised draft:",
        "llm": llm_smart,
        "output_key": "final_content"
    }
}

# ── EXAM GENERATOR ───────────────────────────────────────────────────────────
# Targets Assessment rubric §1 Question Quality + §2 Answer Key + §3–5 shared.
NODE_CONFIGS["exam_generator"] = {
    "system_role": "STEM Assessment Author",
    "system_instruction": (
        "Generate an exam directly aligned to the lab's learning objectives.\n\n"
        "SECTION A — 10 MCQs (rubric §1 + §3):\n"
        "- Each question tests a concept from the lab (not trivia)\n"
        "- All 4 distractors must be plausible (rubric §1: reasonable wrong answers)\n"
        "- Questions are self-contained — no answer depends on another (rubric §1)\n"
        "- Mix of recall, application, and interpretation questions (rubric §4)\n"
        "- Label: Q1–Q10, options A–D\n\n"
        "SECTION B — 5 short-answer questions (rubric §1 + §4):\n"
        "- Difficulty progresses easy → hard (rubric §4)\n"
        "- At least 2 require writing or debugging code (rubric §4 higher thinking)\n"
        "- At least 1 requires geospatial interpretation (CRS, projection, or map)\n"
        "- Label: SQ1–SQ5; state point value per question\n\n"
        "ANSWER KEY (rubric §2):\n"
        "- MCQ: Q1: B format\n"
        "- Short answer: model answer + partial credit criteria where applicable\n"
        "- Every answer includes a 1-sentence justification (rubric §2)\n\n"
        "FORMAT RULES:\n"
        "- Plain text only. No markdown, no backtick fences.\n"
        "- Separate key with exactly: ---ANSWER_KEY_START---\n"
        "- State total points and time estimate at top (rubric §5)"
    ),
    "human_instruction": "Lab content:",
    "llm": llm_smart,
    "output_key": "exam_content"
}


In [ ]:
# 3. Set up the system to make the agent work

def make_messages(config: dict, state: dict, extra: str) -> list:
    """
    Takes a node config, the graph state, and additional text, returning formatted messages for the LLM.
    """
    # 1. Build the System Message
    system_content = f"Role: {config['system_role']}\nInstructions: {config['system_instruction']}"
    system_message = SystemMessage(content=system_content)

    # 2. Build the Human Message
    human_content = f"{config.get('human_role', '')}\nInstructions: {config['human_instruction']}\n\n {extra}"
    human_message = HumanMessage(content=human_content)

    # Tried and failed at thi
    #return [system_content, human_content]

    # 3. Return the exact list format that llm.invoke() wants
    return [system_message, human_message]


In [ ]:
# 4. Make the nodes

def brainstormer_node(state: dict):
    config = NODE_CONFIGS["brainstormer"]

    extra = state['initial_topic']

    # Use our helper to prep the prompt and add information
    messages = make_messages(config, state, extra)
    #messages[1].append({state['initial_topic']})
    #messages[1] += state['initial_topic']
    #msg = [SystemMessage(content=messages[0]),HumanMessage(content=messages[1])]
    #response = config['llm'].invoke(msg)

    # Call the LLM
    response = config['llm'].invoke(messages)

    # Update the state
    return {config['output_key']: [response.content]}

def writer_node(state: dict):
    config = NODE_CONFIGS["writer"]

    extra = f"Topic: {state['initial_topic']} \nBrainstorm ideas: {state['brainstorm']}"

    # Use our helper to prep the prompt and add information
    messages = make_messages(config, state, extra)

    # Call the LLM
    response = config['llm'].invoke(messages)

    # Update the state
    return {config['output_key']: [response.content]}

def reviewer_a(state: dict):
    config = NODE_CONFIGS["reviewer_a"]

    extra = state['draft']

    # Use our helper to prep the prompt and add information
    messages = make_messages(config, state, extra)

    # Call the LLM
    response = config['llm'].invoke(messages)

    # Update the state
    return {config['output_key']: [response.content]}

def reviewer_b(state: dict):
    config = NODE_CONFIGS["reviewer_b"]

    extra = state['draft']

    # Use our helper to prep the prompt and add information
    messages = make_messages(config, state, extra)
    #messages[1].append({state['draft']})

    # Call the LLM
    response = config['llm'].invoke(messages)

    # Update the state
    return {config['output_key']: [response.content]}

def reviewer_c(state: dict):
    config = NODE_CONFIGS["reviewer_c"]

    extra = state['draft']

    # Use our helper to prep the prompt and add information
    messages = make_messages(config, state, extra)
    #messages[1].append({state['draft']})

    # Call the LLM
    response = config['llm'].invoke(messages)

    # Update the state
    return {config['output_key']: [response.content]}


def editor_node(state: dict):
    config = NODE_CONFIGS["editor"]

    all_reviews = "\n\n".join(state['reviewer_feedback'])

    extra = f"Draft: {state['draft']}\n\n Reviews: {all_reviews}"

    # Use our helper to prep the prompt and add information
    messages = make_messages(config, state, extra)

    # Call the LLM
    response = config['llm'].invoke(messages)

    # Update the state
    return {config['output_key']: [response.content]}

def reviser_node(state: dict):
    config = NODE_CONFIGS["reviser"]

    all_reviews = "\n\n".join(state['reviewer_feedback'])

    extra =  f"Draft: {state['draft']}\n\n"
    extra += f"Reviews: {all_reviews}\n\n"
    extra += f"Editor Feedback: {state['editor_feedback']}\n\n"

    # Use our helper to prep the prompt and add information
    messages = make_messages(config, state, extra)

    # Call the LLM
    response = config['llm'].invoke(messages)

    # Update the state
    return {config['output_key']: [response.content]}

def copywriter_node(state: dict):
    config = NODE_CONFIGS["copywriter"]

    extra = f"Revised Draft: {state['revised_draft']}\n\n"

    # Use our helper to prep the prompt and add information
    messages = make_messages(config, state, extra)

    # Call the LLM
    response = config['llm'].invoke(messages)

    # Update the state
    return {config['output_key']: [response.content]}


# ── EXAM GENERATOR NODE ────────────────────────────────────────────────────
def exam_generator_node(state: dict):
    config = NODE_CONFIGS["exam_generator"]
    extra = f"Lab Content:\n{state['final_content']}"
    messages = make_messages(config, state, extra)
    response = config['llm'].invoke(messages)
    return {config['output_key']: response.content}


# ── PDF EXPORTER NODE ──────────────────────────────────────────────────────
def pdf_exporter_node(state: dict):
    """
    Splits exam_content at ---ANSWER_KEY_START--- and writes two PDFs:
      - exam_paper.pdf   : questions only
      - answer_sheet.pdf : MCQ + short-answer key
    Uses reportlab Platypus for clean, professional formatting.
    """
    try:
        from reportlab.lib.pagesizes import letter
        from reportlab.platypus import (
            SimpleDocTemplate, Paragraph, Spacer, HRFlowable, PageBreak
        )
        from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
        from reportlab.lib.units import inch
        from reportlab.lib import colors

        raw = state.get("exam_content", "")

        # ── Split into exam body and answer key ──
        if "---ANSWER_KEY_START---" in raw:
            exam_body, answer_key = raw.split("---ANSWER_KEY_START---", 1)
        else:
            exam_body   = raw
            answer_key  = "Answer key not generated."

        exam_body   = exam_body.strip()
        answer_key  = answer_key.strip()

        # ── Styles ──
        styles = getSampleStyleSheet()
        title_style = ParagraphStyle(
            "ExamTitle",
            parent=styles["Title"],
            fontSize=18,
            spaceAfter=6,
            textColor=colors.HexColor("#1a3a5c"),
        )
        heading_style = ParagraphStyle(
            "SectionHeading",
            parent=styles["Heading2"],
            fontSize=13,
            spaceBefore=14,
            spaceAfter=6,
            textColor=colors.HexColor("#2e6da4"),
        )
        q_style = ParagraphStyle(
            "Question",
            parent=styles["Normal"],
            fontSize=11,
            leading=16,
            spaceBefore=8,
            leftIndent=0,
        )
        opt_style = ParagraphStyle(
            "Option",
            parent=styles["Normal"],
            fontSize=11,
            leading=14,
            leftIndent=20,
        )
        ans_style = ParagraphStyle(
            "Answer",
            parent=styles["Normal"],
            fontSize=11,
            leading=15,
            leftIndent=10,
            spaceBefore=4,
        )
        meta_style = ParagraphStyle(
            "Meta",
            parent=styles["Normal"],
            fontSize=9,
            textColor=colors.grey,
            spaceAfter=4,
        )

        topic = state.get("initial_topic", "GeoAI Lab")

        def build_exam_story(body_text):
            story = []
            story.append(Paragraph(f"Exam: {topic}", title_style))
            story.append(Paragraph("Name: ___________________________   Date: _______________   Score: _______ / 100", meta_style))
            story.append(HRFlowable(width="100%", thickness=1.5, color=colors.HexColor("#1a3a5c"), spaceAfter=10))

            section = None
            for line in body_text.splitlines():
                line = line.strip()
                if not line:
                    story.append(Spacer(1, 6))
                    continue
                if line.upper().startswith("SECTION A"):
                    story.append(Paragraph("Section A — Multiple Choice (40 points)", heading_style))
                    story.append(Paragraph("Circle the letter of the best answer. Each question is worth 4 points.", opt_style))
                    section = "mcq"
                elif line.upper().startswith("SECTION B"):
                    story.append(PageBreak())
                    story.append(Paragraph("Section B — Short Answer (60 points)", heading_style))
                    story.append(Paragraph("Answer each question in 3–5 sentences. Show any code where asked. Each question is worth 12 points.", opt_style))
                    section = "short"
                elif line.startswith("Q") and line[1:3].replace("1","").replace("0","").isdigit() or (len(line) > 2 and line[0] == "Q" and line[1].isdigit() and ":" in line):
                    story.append(Paragraph(line, q_style))
                elif line.startswith("SQ") and ":" in line:
                    story.append(Spacer(1, 4))
                    story.append(Paragraph(line, q_style))
                    # Add answer lines for short answer
                    for _ in range(5):
                        story.append(Paragraph("_" * 90, ans_style))
                elif line and line[0] in "ABCD" and len(line) > 2 and line[1] in ".) ":
                    story.append(Paragraph(line, opt_style))
                else:
                    story.append(Paragraph(line, q_style))
            return story

        def build_answer_story(body_text, key_text):
            story = []
            story.append(Paragraph(f"Answer Sheet: {topic}", title_style))
            story.append(Paragraph("For instructor use only.", meta_style))
            story.append(HRFlowable(width="100%", thickness=1.5, color=colors.HexColor("#c0392b"), spaceAfter=10))

            # MCQ grid
            story.append(Paragraph("Section A — MCQ Answers", heading_style))
            mcq_lines = [l.strip() for l in key_text.splitlines() if l.strip().startswith("Q") and ":" in l and not l.strip().startswith("SQ")]
            if mcq_lines:
                for l in mcq_lines:
                    story.append(Paragraph(l, ans_style))
            else:
                story.append(Paragraph("(See ANSWER KEY in exam content)", ans_style))

            story.append(PageBreak())
            story.append(Paragraph("Section B — Short Answer Model Answers", heading_style))
            sq_lines = [l.strip() for l in key_text.splitlines() if l.strip().startswith("SQ")]
            if sq_lines:
                for l in sq_lines:
                    story.append(Paragraph(l, q_style))
                    story.append(Spacer(1, 8))
            else:
                # dump full key text
                for l in key_text.splitlines():
                    if l.strip():
                        story.append(Paragraph(l.strip(), ans_style))
            return story

        # ── Build exam PDF ──
        exam_path = "exam_paper.pdf"
        doc = SimpleDocTemplate(
            exam_path, pagesize=letter,
            leftMargin=inch, rightMargin=inch,
            topMargin=inch, bottomMargin=inch
        )
        doc.build(build_exam_story(exam_body))

        # ── Build answer sheet PDF ──
        ans_path = "answer_sheet.pdf"
        doc2 = SimpleDocTemplate(
            ans_path, pagesize=letter,
            leftMargin=inch, rightMargin=inch,
            topMargin=inch, bottomMargin=inch
        )
        doc2.build(build_answer_story(exam_body, answer_key))

        print(f"[PDF Exporter] Saved: {exam_path} and {ans_path}")

    except ImportError:
        print("[PDF Exporter] reportlab not installed. Run: pip install reportlab")

    return {}


In [ ]:
# Create a workflow using AgentState
def make_workflow():
    workflow = StateGraph(AgentState)
    return workflow

# Add nodes to our agent workflow
def make_nodes(workflow):
    workflow.add_node("brainstormer", brainstormer_node) # Brainstorming
    workflow.add_node("writer",       writer_node)       # Drafts

    # 3 reviewers
    workflow.add_node("reviewer_a", reviewer_a)
    workflow.add_node("reviewer_b", reviewer_b)
    workflow.add_node("reviewer_c", reviewer_c)

    workflow.add_node("editor",     editor_node)
    workflow.add_node("reviser",    reviser_node)
    workflow.add_node("copywriter",      copywriter_node)
    workflow.add_node("exam_generator",  exam_generator_node)
    workflow.add_node("pdf_exporter",    pdf_exporter_node)

    return workflow

# Define the edges and the flow of the agent workflow
def make_connections(workflow):

    # The start
    workflow.set_entry_point("brainstormer")

    workflow.add_edge("brainstormer", "writer")

    # Multiple reviewers
    # FAN-OUT: Writer goes to all reviewers at once
    workflow.add_edge("writer", "reviewer_a")
    workflow.add_edge("writer", "reviewer_b")
    workflow.add_edge("writer", "reviewer_c")

    # FAN-IN: All reviewers point to the Editor
    # LangGraph automatically waits for ALL THREE to finish before starting the Editor.
    workflow.add_edge("reviewer_a", "editor")
    workflow.add_edge("reviewer_b", "editor")
    workflow.add_edge("reviewer_c", "editor")

    # Bring them all back and revise
    workflow.add_edge("editor", "reviser")
    workflow.add_edge("reviser", "copywriter")

    # The end
    workflow.add_edge("copywriter",     "exam_generator")
    workflow.add_edge("exam_generator", "pdf_exporter")
    workflow.add_edge("pdf_exporter",   END)

    return workflow

def make_app():
    workflow = make_workflow()
    workflow = make_nodes(workflow)
    workflow = make_connections(workflow)

    # Compile the workflow
    app = workflow.compile(checkpointer=memory)

    return app

In [ ]:
# Kick off the process
inputs = {"initial_topic": "", "draft": "",
          "reviewer_feedback": [], "editor_feedback": "",
          "revised_draft": "", "final_content": "",
          "revision_count": 0, "exam_content": ""}
config = {"configurable": {"thread_id": "1"}}

# Set the initial topic
inputs['initial_topic'] = initial_topic

# Make the agent app
app = make_app()

for output in app.stream(inputs, config):
    for key, value in output.items():
        print(f"Finished node: {key}")

# Final Result
final_state = app.get_state(config)
print("\n--- FINAL POLISHED CONTENT ---")
print("FS",final_state)
print("FSV",final_state.values)

final_values = app.get_state(config).values
if "final_content" in final_values:
        content = final_values["final_content"]
#print(final_state.values["final_polish"])

print(final_state.values.keys())
#print(final_state.values['draft'])
print(final_state.values['final_content'])

Finished node: brainstormer
Finished node: writer
Finished node: reviewer_a
Finished node: reviewer_c
Finished node: reviewer_b
Finished node: editor
Finished node: reviser
Finished node: copywriter
Finished node: exam_generator
[PDF Exporter] reportlab not installed. Run: pip install reportlab
Finished node: pdf_exporter

--- FINAL POLISHED CONTENT ---
FS StateSnapshot(values={'initial_topic': 'Masked Auto Encoder (MAE) for GeoAI', 'brainstorm': ["Title: Green Infrastructure Mapping with MAE\nConcept: Develop a Masked Auto Encoder (MAE) to identify and classify green infrastructure features in urban environments using spectral reflectance patterns and spatial context.\nData: Sentinel-2\nCRS: EPSG:32633 (UTM zone 33N, optimal for high-resolution imagery in urban areas worldwide)\nMethod: A custom MAE trained on multispectral bands to reconstruct masked regions and classify vegetation types.\nApplication: Enhance urban planning by accurately assessing green infrastructure distribution.\

In [ ]:
with open("output.txt", "w") as file:
    file.write(str(final_state.values['final_content']))

In [ ]:
import nbformat

def simple_text_to_jupyter(raw_text, output_filename="geospatial_lab.ipynb"):

    raw_text = raw_text.replace('\\n', '\n')
    raw_text = raw_text.replace('\\\'', '\'')

    # 1. Create a blank notebook using the official library
    nb = nbformat.v4.new_notebook()

    # 2. THE TRICK: Replace the markers with a unique splitting word
    text = raw_text.replace("Markdown Cell", "SPLIT_HERE_MARKDOWN")
    text = text.replace("Code Cell", "SPLIT_HERE_CODE")

    # 3. Break the text into a list of chunks based on our unique word
    chunks = text.split("SPLIT_HERE_")

    for chunk in chunks:
        # 4. If it's a markdown chunk, clean the text and add a markdown cell
        if chunk.startswith("MARKDOWN"):
            content = chunk.replace("MARKDOWN", "").strip()
            if content:
                nb.cells.append(nbformat.v4.new_markdown_cell(content))

        # 5. If it's a code chunk, clean the text and add a code cell
        elif chunk.startswith("CODE"):
            content = chunk.replace("CODE", "#").strip()

            # (Optional but helpful) strip out backticks if the LLM added them
            content = content.replace("```python", "").replace("```", "").strip()

            if content:
                nb.cells.append(nbformat.v4.new_code_cell(content))

    # 6. Save the file using the library's built-in writer
    with open(output_filename, "w", encoding="utf-8") as f:
        nbformat.write(nb, f)

    print(f"Saved to {output_filename} successfully!")

final_text = str(final_state.values['final_content'])

simple_text_to_jupyter(final_text)

Saved to geospatial_lab.ipynb successfully!


## Exam PDF Output
After running the pipeline, two PDF files are generated in the working directory:

In [ ]:
# Display paths to generated PDFs
import os
for fname in ["exam_paper.pdf", "answer_sheet.pdf"]:
    if os.path.exists(fname):
        print(f"Generated: {os.path.abspath(fname)}")
    else:
        print(f"Not found: {fname} — check that the pipeline ran successfully.")


Not found: exam_paper.pdf — check that the pipeline ran successfully.
Not found: answer_sheet.pdf — check that the pipeline ran successfully.
